# Lab 2 財務資料清理：把髒表格變成模型能吃的特徵

**今天的目標：** 把上一個 Lab 抓回來的**髒股價資料洗乾淨**，再做成模型可以直接吃的特徵表。你會走過四段：

- **A** 修 `prices.csv`：民國日期、千分位逗號字串 → 能算數學的資料
- **B** 寬表與缺值：三種補法，**其中一種是偷看未來**
- **C** ⭐ 特徵工廠 + 第一次 train/test **時序分割**（下一個 Lab 的 KNN 直接吃這張表）
- **D** 收尾

> 🚫 **免責：** 今天整理的股價、做出的「明日漲跌」標籤，純為教學示範資料怎麼從髒變乾淨，**不是投資建議**。
> 🌱 **今天所有數字都跟教材一模一樣**（固定的資料檔＋固定的亂數種子）。⚠️ **這跟上一個 Lab 相反**——上次抓活網站，「跟老師不一樣＝正常」；**今天跟老師不一樣＝有地方跑掉了**，請回頭檢查。

In [ ]:
# 📦 先跑這一格：一次裝齊今天要用的套件（已裝好的會直接跳過；裝不起來看 README）
!pip install -q pandas numpy requests

In [ ]:
# 老師的小設定：關掉一個無關緊要的套件提醒，讓等一下的輸出乾淨一點（直接跑、不用改）
import warnings
warnings.filterwarnings("ignore")

## 🔧 第 0 步：環境檢查

**預期輸出：** 印出 `pandas` 和 `numpy` 的版本號——有版本號就代表套件裝好了。

In [ ]:
import pandas as pd
import numpy as np
print("pandas", pd.__version__)
print("numpy ", np.__version__)

---
## A・修 `prices.csv`：上一個 Lab 留下的兩個髒點

上一個 Lab 你把證交所股價抓回 `prices.csv`，當時老師指了兩個地方沒修：
① 日期是**民國紀年字串** `'115/06/01'`；② 數字是**含千分位逗號的字串** `'60,942,792'`。**今天就是修它們的時候。**

今天用教材附的 `data/prices.csv`（證交所 2330、2026 年 6 月）——同一支 API、同一檔股票，只是換個月份，**髒點一模一樣**。你自己抓的那份請留著，它證明你真的抓得到。

> 💬 統一用同一份檔案，你才能拿自己的輸出跟教材**逐字對答案**。

### A1・照妖鏡：`dtypes` 看每一欄到底是什麼型別

`dtype`（資料型別）＝pandas 眼中「這一欄裝的是什麼」：`int64` 整數、`float64` 小數、`datetime64` 日期、**`object` ＝文字**。

**預期輸出：** 除了「註記」，**每一欄都是 `object`**——連看起來像數字的開盤價、收盤價都是**文字**。

In [ ]:
import pandas as pd

df = pd.read_csv("data/prices.csv")   # 💡 教材附的證交所快照（2330 / 2026-06）
print(df.dtypes)                       # 💡 一行照妖：長得像數字 ≠ 是數字

### A2・先嚐痛點：叫文字欄算平均會怎樣

**預期輸出：** 一大片紅色的 `TypeError`。

> ⚠️ **這格故意讓它出錯**，不是你打錯字。跑下去，看清楚錯誤訊息長什麼樣——之後你自己寫程式碰到它時才認得出來。

In [ ]:
df["成交股數"].mean()      # 字串欄叫它算平均 → 爆！
# 💡 這個錯不是 bug，是 pandas 在保護你：它不猜你想怎麼轉，要你自己講清楚

### A3・修第一個髒點：民國轉西元

證交所給的是 `'115/06/01'`（民國 115 年 ＝ 西元 2026 年，**+1911**）。

要把整欄 21 個日期都轉掉，笨方法是寫個 `for` 迴圈一格一格處理。但 pandas 有更省事的做法：**寫一個「處理一格」的小函式，然後叫 pandas 幫你套到每一格。**

**預期輸出：** `2026-06-01 00:00:00`，型別變成 `datetime64[ns]`。

In [ ]:
def roc_to_date(s):                       # '115/06/01' → Timestamp('2026-06-01')
    y, m, d = s.split("/")                # 💡 拆成 ['115', '06', '01']
    return pd.Timestamp(int(y) + 1911, int(m), int(d))

# TODO：把上面寫好的 roc_to_date 套到「日期」這一整欄。
#       要的不是 for 迴圈——pandas 有一個方法可以「對這欄的每一格，套同一個函式」。
df["日期"] = df["日期"].____(roc_to_date)
print(df["日期"].iloc[0])
print(df["日期"].dtype)                    # 對了的話型別會從 object 變成 datetime64

### A4・修第二個髒點：拔逗號 + 轉型別

`'60,942,792'` 含逗號，**不是合法的數字字串**，直接轉會爆。跟 A3 一樣的套路：**寫一個「處理一格」的小函式 → 套到整欄。**

> ⚠️ 順序很重要：**先拔逗號、再轉型別**。反過來會爆。

**預期輸出：** 日期 `datetime64[ns]`、成交股數 `int64`、收盤價 `float64`。

In [ ]:
# TODO：補完這兩個「處理一格」的小函式。
#       ⚠️ 拿到的是字串 '60,942,792'：要先把逗號拿掉，才轉得成數字。順序反了會爆。
def to_int(s):                            # '60,942,792' → 60942792
    return int(s.____(",", ""))

def to_float(s):                          # '2,355.00' → 2355.0
    return float(s.____(",", ""))

# TODO：跟 A3 一樣，把函式套到整欄（同一個方法）
for col in ["成交股數", "成交金額", "成交筆數"]:
    df[col] = df[col].____(to_int)        # 股數/金額/筆數是整數（不會有 0.5 股）

for col in ["開盤價", "最高價", "最低價", "收盤價"]:
    df[col] = df[col].____(to_float)      # 價格有小數

print(df.dtypes[["日期", "成交股數", "收盤價"]])

### A5・驗收：剛剛爆錯的那行，現在算得出來了

**預期輸出：** `43224891`

In [ ]:
print("這個月平均每天成交股數：", round(df["成交股數"].mean()))
# 💡 同一份資料，洗乾淨之前模型連碰都碰不了——這就是資料清理的價值
# 💡 round()＝四捨五入到整數（平均算出來會有一長串小數，這裡不需要）

### 📝 小作業 A

資料修好之後，`describe()` 一行就能看完一整欄的全貌。

`describe()` 會吐 8 個數字：
- **count** 有幾筆　**mean** 平均　**std** 標準差（數字跳得兇不兇）
- **min / max** 最小 / 最大
- **25% / 50% / 75%** ＝把資料由小到大排好後，站在 1/4、一半、3/4 位置的那個值（**50% 就是中位數**）

1. 用 `describe()` 看「收盤價」這個月的全貌。
2. ⭐ 想一想：`mean`（平均）和 `50%`（中位數）差多少？差很多代表什麼？

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
print(df["收盤價"].describe())
```
**結論：** count 21（21 個交易日）、min 2250 / max 2510、mean 約 2371、50% 是 2380。
mean 和中位數很接近 → 這個月**沒有被極端值拉走**，價格分布算平均。
如果哪天你看到 mean 遠大於中位數，通常代表**有幾個極大值把平均拉高了**——這在成交量、市值這種欄位很常見。
</details>

In [ ]:
# 📝 小作業 A：你的答案

# TODO：用一個方法看完「收盤價」的全貌（count / mean / std / min / 25% / 50% / 75% / max）
print(df["收盤價"].____())

---
## B・寬表與缺值：三種補法，其中一種在作弊

A 段修的是「一檔股票的一個月」。真實的量化分析常常是「**很多檔股票 × 很多天**」——那張表叫**寬表**：**每一列是一天、每一欄是一檔股票**。

**寬表為什麼會有缺值（`NaN`）？** 因為每一列是「**全市場的那一天**」。某檔那天沒交易、別的有 → **它那格就是 `NaN`**。
所以：**全市場休市**（大家都沒開）不產生 `NaN`；**只有你這檔沒開**（停牌）才會。

### B1・生一張教學用的寬表

用固定種子生 60 天 × 5 檔的**虛構**行情，並刻意做出兩種缺值劇情。

**預期輸出：** `(60, 5)`，戊的前兩天是 `NaN`。

> ⚠️ 甲乙丙丁戊和這些價格**全是亂數**，不是真實行情。但「晚上市」「停牌」這兩種劇情是真的——**B6 會用證交所的資料驗證給你看**。

In [ ]:
import numpy as np
import pandas as pd

def make_panel():
    rng = np.random.default_rng(42)                    # 🌱 固定種子：你跑出來跟教材一模一樣
    dates = pd.bdate_range("2020-01-01", periods=60)   # 💡 bdate_range＝只含工作日（股市週末不開盤）
    codes = ["甲", "乙", "丙", "丁", "戊"]              # 5 檔【虛構】股票
    walk = 100 + np.cumsum(rng.normal(0, 1, size=(60, 5)), axis=0)   # 💡 隨機漲跌累加＝走出來的假股價
    panel = pd.DataFrame(walk.round(2), index=dates, columns=codes)
    panel.loc[dates[:20], "戊"] = np.nan               # 戊：前 20 天「還沒上市」
    panel.loc[dates[30:35], "丙"] = np.nan             # 丙：中段「停牌」5 天
    print("生成教學寬表", panel.shape)
    return panel

panel = make_panel()
print(panel.head(2))

### B2・先看每一格是不是 `NaN`

**預期輸出：** 一整張 `True` / `False` 的表——`True` 就是那一格沒資料。

In [ ]:
# TODO：逐格問「這一格是不是 NaN」。回來的會是一張同樣大小的 True / False 表。
print(panel.____())

### B3・再把它加總，看每檔缺幾天

`True` 在加總時算 **1**、`False` 算 **0**——所以直接 `.sum()` 就是「每欄有幾個 `NaN`」。

**預期輸出：** 丙缺 5 天、戊缺 20 天，其他都是 0。

> 盯著這兩個數字：**它們背後是兩種完全不同的故事。**

In [ ]:
# TODO：接續上一格，把每一欄的 True 加總起來（True 算 1、False 算 0）
print(panel.____().____())

### B4・第一種處理：有缺就整列丟

**預期輸出：** 60 天只剩 35 天。

In [ ]:
print("原本：", panel.shape[0], "天")
# TODO：把「只要那一列有任何缺值，整列丟掉」的方法填進來
print("整列丟掉後剩：", panel.____().shape[0], "天")
# 想一想：丟掉的那 25 天裡，甲乙丁其實是完全健康的資料

### B5・⭐ 課程重點：停牌那天，你要拿什麼數字去填？

丙在 `2020-02-14` 停牌中，**沒有成交價**。接下來三格，我們用兩種方式填填看，然後回頭查這兩個數字**各是從哪裡來的**。

先看原值。

**預期輸出：** `nan`

In [ ]:
day = "2020-02-14"                                    # 停牌窗是 make_panel 裡我們自己設的 dates[30:35]

print("原值：", panel.loc[day, "丙"])                   # NaN（停牌沒有成交價）

### B5-1・用「前面」的值往後填

**預期輸出：** `93.51`

In [ ]:
# TODO：用「前面」的值往後填（forward fill）
print("用昨天補：", panel.____().loc[day, "丙"])

### B5-2・用「後面」的值往前填

**預期輸出：** `95.03`

> 兩個數字都拿到了。**先別急著往下，猜猜看哪一個有問題。**

In [ ]:
# TODO：用「後面」的值往前填（backward fill）
print("用明天補：", panel.____().loc[day, "丙"])

### B5-3・揭曉：這兩個數字各是從哪來的

**預期輸出：** 停牌前是 `93.51`、復牌後是 `95.03`——**跟剛剛那兩個數字一模一樣。**

In [ ]:
print("停牌前最後一價（2020-02-11）：", panel.loc["2020-02-11", "丙"])
print("復牌後第一價  （2020-02-19）：", panel.loc["2020-02-19", "丙"])
# 💡 用昨天補填的 93.51 ＝ 停牌前最後成交價 → 「市場最後一次對它的定價」，合理 ✓
# 💡 用明天補填的 95.03 ＝ 復牌後第一個價 → 2/14 那天全世界沒人知道這個數字，這叫偷看未來 ✗

### B6・`ffill` 補不了「上市前」——這是對的

**預期輸出：** `20`（戊前 20 天用「前面的值」補完之後，仍然是 `NaN`）。

> ⚠️ 這不是 bug。往下看註解。

In [ ]:
panel_filled = panel.ffill()
print("ffill 後戊還缺幾天：", panel_filled["戊"].isna().sum())
# 💡 ffill 是「用前面的值帶下來」，戊上市前前面根本沒有值可以帶
# 💡 而且上市前本來就沒有價格，硬塞一個假價格反而是捏造資料 → 留著 NaN 才對

### B7・⭐ 回證交所驗證：停牌不是我編的

你可能已經在想：「這個停牌該不會是老師為了上課編出來的吧？」——我們用**上一個 Lab 學過的同一支證交所 API**，抓真實的鴻海（2317）看看。

**預期輸出：** 台積電 22 天、鴻海 16 天，差的正是 `107/10/18` ~ `107/10/25` 那 6 天。

> ⚠️ **這格需要網路**（今天唯一需要網路的地方）。連不上就跳過，**不影響後面任何一步**。

In [ ]:
import requests
import time

def trading_days(stock_no):                    # 回傳「這檔股票這個月有交易的日期」
    r = requests.get("https://www.twse.com.tw/exchangeReport/STOCK_DAY",
                     params={"response": "json", "date": "20181001", "stockNo": stock_no})
    j = r.json()
    df_month = pd.DataFrame(j["data"], columns=j["fields"])
    return set(df_month["日期"])                # 💡 set＝集合，等一下要拿兩個集合相減

tsmc = trading_days("2330")                    # 台積電
time.sleep(1)                                  # 💡 守禮貌頻率，跟上一個 Lab 一樣
hon = trading_days("2317")                     # 鴻海

only_tsmc = sorted(tsmc - hon)                 # 💡 集合相減＝「在台積電裡、但不在鴻海裡」的日期
print("台積電 2018/10 交易天數：", len(tsmc))
print("鴻海   2018/10 交易天數：", len(hon))
print("台積電有、鴻海沒有的日子：", only_tsmc)
# 💡 集合相減自動濾掉了休市日：10/10 國慶日兩檔都沒開，所以不會出現在差集裡
# 💡 那 6 天鴻海現金減資 20%，舊股 10/17 最後交易、新股 10/26 上市——真實世界的「停牌」

> **🔑 同一個月、同一個市場，台積電交易 22 天、鴻海只有 16 天。** 把這兩檔放進一張寬表，鴻海那 6 格就是 `NaN`——**跟你剛剛在「丙」身上練的一模一樣，只是這次是真的。**
>
> ⚠️ **順帶一個誠實提醒（本課不深入，但你要知道它存在）：** 鴻海停牌前 68.10、復牌後 76.20，那**不是漲了 12%**，是「一仟股換 800 股」的減資換算。真實金融資料除了缺值，還有這一層失真要處理，那是量化的進階題。

### 📝 小作業 B

1. 分別用兩種補法套在**整張 panel** 上，數數看各自還剩幾個 `NaN`。想想看：為什麼有一種會剩、有一種是 0？

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
print("原本總共幾個 NaN：", panel.isna().sum().sum())
print("ffill 後：", panel.ffill().isna().sum().sum())
print("bfill 後：", panel.bfill().isna().sum().sum())
```
**結論：** ffill 後剩 20（戊上市前補不了，**這是對的**）；bfill 後是 0——它把未來的價往回填，**看起來最乾淨，其實在作弊**。
**「把 NaN 清光」不是目標，「只補補得有道理的」才是。**
</details>

In [ ]:
# 📝 小作業 B：你的答案

print("原本總共幾個 NaN：", panel.isna().sum().sum())   # sum().sum()＝先每欄加總、再全部加總
# TODO：填入兩種補法，比較各自還剩幾個 NaN
print("用昨天補之後：", panel.____().isna().sum().sum())
print("用明天補之後：", panel.____().isna().sum().sum())
# 想一想：為什麼有一種會剩、有一種是 0？剩下的那個是 bug 嗎？

---
## C・⭐ 特徵工廠 + 第一次時序分割

這段做出下一個 Lab 的 KNN 可以**直接吃**的 (X, y)，並第一次拿出「**樣本內 vs 樣本外**」這把尺。

三個新詞：
- **報酬率**：今天比昨天漲跌幾 %。100→101 和 50→50.5 是同一件事（+1%），**報酬率才能跨股票比較**。
- **移動平均**：最近 N 天的平均價，把每天亂跳的雜訊抹平。**只用過去的資料。**
- **label（標籤）**：要模型猜的答案。我們做「明日漲」＝明天收盤比今天高就是 1。

### C1・做兩個特徵：報酬率 + 5 日移動平均

**預期輸出：** 前幾列的 `ret1` / `ma5` 是 `NaN`——**這是正常的**，往下看註解。

In [ ]:
s = panel_filled["甲"]                     # 用 ffill 後的「甲」單檔（虛構）

# TODO：補上兩個特徵的算法
feat = pd.DataFrame({
    "close": s,                            # 原始收盤價（留著對照）
    "ret1":  s.____(),                     # 一日報酬率：今天比昨天漲跌幾 %（pandas 有現成方法）
    "ma5":   s.____(5).mean(),             # 5 日移動平均：開一個「往回看 5 天」的窗，再取平均
})
print(feat.head(6))
# 開頭會有一排 NaN——先想想為什麼，下一格會講

### C2・做 label「明日漲」，先直接看結果

**預期輸出：** 一整欄 0 / 1。

> ⚠️ **特別注意最後一列。** 資料的最後一天哪來的「明天」？下一格處理它。

In [ ]:
# TODO：做出「明天收盤比今天高 → 1，否則 0」。
#       提示：你需要一個「把整欄往上挪一格」的方法，讓每一列旁邊放的是【明天】的價，
#       再拿它跟今天比大小。⚠️ 挪的方向弄反就變成「昨天」了。
feat["明日漲"] = (____).astype(int)

feat["明日漲"]

### C3・丟掉兩種「算不出來」的列

**預期輸出：** `(55, 4)`——60 天 − 開頭 4 天特徵算不出來 − 最後 1 天沒有「明天」＝ 55 列。

In [ ]:
feat = feat.dropna()                              # 丟掉開頭特徵算不出來的那幾列
feat = feat.iloc[:-1]                             # ⭐ 丟掉最後一天

print(feat.shape)
print(feat.head(2))
# 💡 為什麼一定要丟最後一天？資料的最後一天沒有「明天的價」可比，shift(-1) 那格是 NaN，
#    astype(int) 會把它變成一個假的 0——看起來像「明天沒漲」，其實是「不知道」。
#    一門教誠實衡量的課，不能留一個假答案在表裡。

### C4・⭐⭐ 最重要的一條分界：`y` 看未來 ✓、`X` 看未來 ✗

B 段才說「用明天的價＝偷看未來」，這裡 label 卻光明正大拿明天的價——**矛盾嗎？不矛盾：**

- **label（y）就是要預測的答案**——訓練時本來就要告訴模型「歷史上這天的明天漲了沒」。**答案欄寫答案 ✓**
- **特徵（X）是預測當下看得到的線索**——你站在「今天」，**只能用今天以前的資訊**。X 裡混進明天才知道的數字＝**抄答案 ✗**

> **檢查法（帶得走）：** 每個特徵問一句「**我站在這一天，這個數字我當天算得出來嗎？**」

In [ ]:
# 拿今天做的三個東西，自己驗一遍
for name in ["ret1（今天比昨天）", "ma5（最近 5 天平均）", "明日漲（明天的價）"]:
    print(name)
# 💡 ret1、ma5 都只用今天和之前的資料 → 站在今天算得出來 ✓ 可以當特徵
# 💡「明天的價」站在今天算不出來 ✗ → 只能當 label（答案欄），放進 X 就是作弊

### C5・⭐ 第一次 train/test 時序分割

模型好不好，要用**它沒看過的資料**考。但時間序列多一條鐵則：**要按時間切，前段訓練、後段測試。**

**預期輸出：** train 44 天、test 11 天，最後一行 `True`。

> ⚠️ **絕不可以**隨機打散再切——那等於「用 3 月訓練、回頭考 1 月」，**搭時光機作弊**。

In [ ]:
cut = int(len(feat) * 0.8)                        # 前 80% 訓練、後 20% 測試

# TODO：用 cut 把 feat 切成前後兩段（前段訓練、後段測試）。
#       ⚠️ 兩段不可以重疊，也不可以漏掉中間任何一天。
train, test = feat.iloc[____], feat.iloc[____]

print("train（念書）：", len(train), "天　", train.index[0].date(), "~", train.index[-1].date())
print("test （考試）：", len(test), "天　", test.index[0].date(), "~", test.index[-1].date())
print("訓練全部在測試之前？", train.index.max() < test.index.min())
# 最後這行是「時序切」的不變量：切對了永遠是 True

### 📝 小作業 C

1. 把移動平均的視窗從 5 天改成 10 天，重做一次特徵表，看 `dropna` 後剩幾列。
2. ⭐（有時間再做）把切點 `0.8` 改成 `0.7`，印出新的 train/test 天數——**切點變了，但「訓練全部在測試之前」必須仍然是 `True`。**

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
feat10 = pd.DataFrame({"close": s, "ret1": s.pct_change(), "ma5": s.rolling(10).mean()})
feat10["明日漲"] = (s.shift(-1) > s).astype(int)
feat10 = feat10.dropna().iloc[:-1]
print("rolling(10) →", feat10.shape)      # (50, 4)：開頭多吃掉 5 天
```
**結論：** 移動平均的視窗開得越長，開頭要丟掉的列就越多——**特徵不是免費的，每加一個都要付資料的代價**。
第 2 題不管切點怎麼換，`train.index.max() < test.index.min()` 都必須是 `True`。
</details>

In [ ]:
# 📝 小作業 C-1：你的答案

# TODO：把移動平均的視窗改成 10 天
feat10 = pd.DataFrame({"close": s, "ret1": s.pct_change(), "ma5": s.____(10).mean()})
feat10["明日漲"] = (s.shift(-1) > s).astype(int)
feat10 = feat10.dropna().iloc[:-1]
print("rolling(5)  →", feat.shape)
print("rolling(10) →", feat10.shape)      # 比比看差幾列

### 📝 小作業 C-2（⭐ 進階選做）：換個切點，不變量還在嗎？

In [ ]:
# 📝 小作業 C-2：你的答案

# TODO：把切點改成 0.7，重切一次，再檢查一次不變量
cut7 = int(len(feat) * ____)
train7, test7 = feat.iloc[:cut7], feat.iloc[cut7:]
print("切 0.7 → train", len(train7), "天 / test", len(test7), "天")
print("訓練全部在測試之前？", train7.index.max() < test7.index.min())   # 這裡必須還是 True

---
## 🔑 收尾：今天的交付物 + 一把要用到課程結束的尺

- ✅ 洗乾淨的 `prices.csv`（民國日期 → 西元、逗號字串 → 數字）
- ✅ `(55, 4)` 特徵表：`close` / `ret1` / `ma5` ＋ label `明日漲`
- ✅ 按時間切好的 `train`（44 天）/ `test`（11 天）——**下一個 Lab 的 KNN 直接吃它**

🧭 **比檔案更值錢的：**

1. **「爬回來 → 洗乾淨 → 才能建模」** ——資料工作八成時間花在清理，這是資料工程師的日常。
2. **補缺值也會作弊**：停牌用「昨天」補 ✓、上市前留 `NaN` ✓、用「明天」補 ✗。
3. **`y` 看未來 ✓、`X` 看未來 ✗**，檢查法：「我站在這一天，這個數字當天算得出來嗎？」
4. **有時間軸的資料，切割一律按時間、絕不隨機打散**，不變量是 `train.index.max() < test.index.min()`。

> 第 2~4 點是同一把「**誠實衡量**」的尺。從今天開始，這把尺會一路量到課程結束——下一個 Lab 量 KNN 的 k、再下一個量決策樹的深度。
>
> 🚫 **再講一次免責：** 今天的寬表是亂數假行情，做出來的「明日漲」label 純為教學。**學到的真本事是「把髒資料變成乾淨特徵，而且從第一天就用誠實的方式切資料」，不是預測明天哪檔會漲。**

---
## 🛟 Backup：環境自我檢查

如果今天有哪一格跑不動，用這三行確認 pandas 的核心功能正常。

**預期輸出：** `[10.0, 10.0, 12.0]` / `[10.0, 12.0, 12.0]` / `[nan, 0.0, 0.2]`

In [ ]:
s_check = pd.Series([10.0, np.nan, 12.0], index=pd.bdate_range("2020-01-01", periods=3))
print(s_check.ffill().tolist())                     # 缺值用「昨天」補
print(s_check.bfill().tolist())                     # 缺值用「明天」補
print(s_check.ffill().pct_change().round(4).tolist())   # 💡 先明確 ffill，再算報酬率
# 💡 為什麼第三行要先寫 .ffill()？因為 pct_change() 直接吃有缺值的序列時，
#    pandas 會偷偷幫你 ffill 再算——這個隱藏預設已被標記淘汰，未來版本會拿掉。
#    今天整堂在教「補值方式要自己明確選」，這裡當然要把它寫出來。